##### Setup

In [9]:
from config import client, MODEL
from data_loader import load_documents
from search_engine import create_index, search
from rag import build_context, build_prompt, rag, INSTRUCTIONS
from tools import search_tool
from agent import agent_loop, instructions

##### Load Documents

In [10]:
documents = load_documents()

len(documents)

1368

##### Inspect One Document

In [11]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

##### Build Search Index

In [12]:
index = create_index(documents)

##### Test Search

In [5]:
question = "I just discovered the course. Can I still join now?"

search_results = search(index, question)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'I missed the first homework - can I still get a certificate?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

##### Build Context

In [14]:
context = build_context(search_results)

print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

##### Build Prompt

In [15]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

##### Call The LLM Directly

In [16]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = client.responses.create(
    model=MODEL,
    input=message_history
)

response.output_text

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

##### Check Token Usage

In [17]:
response.usage

ResponseUsage(input_tokens=731, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=33, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=764)

##### Test Plain RAG

In [6]:
answer = rag(question, index)

print(answer)

Yes, you can still join the course now. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.


##### Test Agentic RAG

In [7]:
answer = agent_loop(index, instructions, question)

print(answer)

iteration #1...
function_call: search {"query":"join course late enrollment"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.

If you have any other questions or areas you'd like to explore, feel free to ask!
Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.

If you have any other questions or areas you'd like to explore, feel free to ask!


##### Function Calling

In [18]:
messages = [
    {"role": "user", "content": question}
]

response = client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='call_bSwYPciA9e6JSRqmXyzDesK2', name='search', type='function_call', id='fc_03ad393f4446e27a006a44f970a3588192ba8060ac9f61e269', namespace=None, status='completed')]

##### Test Agentic RAG

In [8]:
answer = agent_loop(index, instructions, question)

print(answer)

iteration #1...
function_call: search {"query":"can I still join the course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted. 

If you have any other questions or areas you'd like to explore, feel free to ask!
Yes, you can still join the course! However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted. 

If you have any other questions or areas you'd like to explore, feel free to ask!


##### Try A Typo Question

In [19]:
answer = agent_loop(index, instructions, "How do I run Olama locally?")

print(answer)

iteration #1...
function_call: search {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
You can run the course locally instead of using Codespaces. Here’s what you need to know:

1. **Environment Setup**: You need to be comfortable setting up Python, `uv`, Jupyter, Docker, and any other necessary tools for the module.

2. **Documentation**: Make sure to document your setup to keep your environment reproducible.

If you're using a specific tool like Olama, ensure that it is compatible with your local setup and follow any additional configurations required for that specific tool.

Is there anything else you want to explore?
You can run the course locally instead of using Codespaces. Here’s what you need to know:

1. **Environment Setup**: You need to be comfortable setting up Python, `uv`, Jupyter, Docker, and any other necessary tools for the module.

2. **Documentation**: Make sure to document your setup to keep your environment reproducible.

If you're using a specific tool lik

##### Plain RAG vs Agentic RAG

Plain RAG:
- developer controls the flow
- search happens once
- fixed pipeline

Agentic RAG:
- LLM decides when to search
- tool calls can happen multiple times
- better for questions that need correction or exploration

##### Frameworks Note

In this project, we built the agentic loop manually.

Frameworks like LangChain, LangGraph, PydanticAI, ToyAIKit, and OpenAI Agents SDK can hide this loop, but the core idea is the same:

model response -> tool call -> tool result -> next model response

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = search(index, question)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'How should I start the course and follow the weekly workflow?']

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = search(index, question)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'How should I start the course and follow the weekly workflow?']

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = search(index, question)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'How should I start the course and follow the weekly workflow?']

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = search(index, question)

[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'How should I start the course and follow the weekly workflow?']